# HPRC Ensembl vs CAT Annotation Comparison - Results Analysis

This notebook aggregates and analyzes results from the Ensembl vs CAT comparison pipeline.

**Input:** TSV files from Nextflow pipeline  
**Output:** Summary statistics, plots, and aggregated tables

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import glob
from pathlib import Path
import numpy as np

# Configure plotting
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

# Paths
RESULTS_DIR = '../nextflow/pipelines/ensembl_cat_comparison/results/results'
OUTPUT_DIR = './analysis_output'
Path(OUTPUT_DIR).mkdir(exist_ok=True)

print("Setup complete!")

: 

## 1. Load All RBH Results

In [ ]:
# Find all RBH files
rbh_files = glob.glob(f'{RESULTS_DIR}/*/*.gene_pairs_rbh.tsv')
print(f"Found {len(rbh_files)} RBH result files")

if len(rbh_files) == 0:
    print(f"ERROR: No files found in {RESULTS_DIR}")
    print("Please check that the Nextflow pipeline has completed successfully.")
else:
    print("\nFirst 5 files:")
    for f in rbh_files[:5]:
        print(f"  {Path(f).parent.name}")

In [ ]:
# Load and combine all RBH data
all_rbh = []
failed = []

for f in rbh_files:
    try:
        assembly = Path(f).parent.name
        df = pd.read_csv(f, sep='\t')
        
        # Check if file is empty or malformed
        if len(df) < 2:  # Header + at least 1 row
            print(f"WARNING: {assembly} has {len(df)} rows (empty or failed)")
            failed.append(assembly)
            continue
        
        df['assembly'] = assembly
        all_rbh.append(df)
        
    except Exception as e:
        print(f"ERROR loading {Path(f).parent.name}: {e}")
        failed.append(Path(f).parent.name)

if all_rbh:
    master_rbh = pd.concat(all_rbh, ignore_index=True)
    print(f"\n✓ Loaded {len(master_rbh):,} RBH pairs from {len(all_rbh)} assemblies")
    print(f"✗ Failed to load {len(failed)} assemblies: {failed[:5]}..." if failed else "✓ All assemblies loaded successfully")
else:
    print("ERROR: No data could be loaded")
    master_rbh = None

In [ ]:
# Quick peek at the data
if master_rbh is not None:
    print("Columns:", master_rbh.columns.tolist())
    print("\nFirst few rows:")
    display(master_rbh.head())
    
    print("\nData types:")
    display(master_rbh.dtypes)

## 2. Per-Assembly Summary Statistics

In [ ]:
# Calculate summary stats per assembly
summary = master_rbh.groupby('assembly').agg({
    'ensembl_id': 'count',
    'is_name_match': 'sum',
    'frac_ensembl_covered': 'mean',
    'frac_cat_covered': 'mean',
}).rename(columns={
    'ensembl_id': 'n_rbh_pairs',
    'is_name_match': 'n_name_matches'
})

# Add classification counts
classification_counts = master_rbh.groupby(['assembly', 'classification']).size().unstack(fill_value=0)
summary = summary.join(classification_counts)

# Calculate percentages
if 'Full-length concordant' in summary.columns:
    summary['pct_full_length'] = (summary['Full-length concordant'] / summary['n_rbh_pairs'] * 100)
summary['pct_name_match'] = (summary['n_name_matches'] / summary['n_rbh_pairs'] * 100)

# Sort by concordance
summary = summary.sort_values('pct_full_length', ascending=False)

print("Per-Assembly Summary Statistics:")
print("=" * 80)
display(summary.head(10))

print("\nOverall Statistics:")
print("=" * 80)
display(summary.describe())

In [ ]:
# Save summary table
summary.to_csv(f'{OUTPUT_DIR}/summary_per_assembly.tsv', sep='\t')
print(f"✓ Saved summary to {OUTPUT_DIR}/summary_per_assembly.tsv")

## 3. Distribution Plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: RBH pairs distribution
axes[0, 0].hist(summary['n_rbh_pairs'], bins=50, edgecolor='black', color='steelblue')
axes[0, 0].set_xlabel('Number of RBH Pairs', fontsize=11)
axes[0, 0].set_ylabel('Number of Assemblies', fontsize=11)
axes[0, 0].set_title('RBH Pairs per Assembly', fontsize=13, fontweight='bold')
axes[0, 0].axvline(summary['n_rbh_pairs'].mean(), color='red', linestyle='--', 
                    label=f"Mean: {summary['n_rbh_pairs'].mean():.0f}")
axes[0, 0].legend()

# Plot 2: Full-length concordance
if 'pct_full_length' in summary.columns:
    axes[0, 1].hist(summary['pct_full_length'], bins=50, edgecolor='black', color='green')
    axes[0, 1].set_xlabel('% Full-length Concordant', fontsize=11)
    axes[0, 1].set_ylabel('Number of Assemblies', fontsize=11)
    axes[0, 1].set_title('Full-length Concordance Distribution', fontsize=13, fontweight='bold')
    axes[0, 1].axvline(summary['pct_full_length'].mean(), color='red', linestyle='--',
                        label=f"Mean: {summary['pct_full_length'].mean():.1f}%")
    axes[0, 1].legend()

# Plot 3: Coverage correlation
axes[1, 0].scatter(summary['frac_ensembl_covered'], summary['frac_cat_covered'], 
                    alpha=0.6, s=50, color='purple')
axes[1, 0].set_xlabel('Mean Ensembl Coverage', fontsize=11)
axes[1, 0].set_ylabel('Mean CAT Coverage', fontsize=11)
axes[1, 0].set_title('Coverage Correlation', fontsize=13, fontweight='bold')
axes[1, 0].plot([0, 1], [0, 1], 'r--', alpha=0.5, label='Perfect correlation')
axes[1, 0].legend()

# Plot 4: Name match rate
axes[1, 1].hist(summary['pct_name_match'], bins=50, edgecolor='black', color='orange')
axes[1, 1].set_xlabel('% Name Matches', fontsize=11)
axes[1, 1].set_ylabel('Number of Assemblies', fontsize=11)
axes[1, 1].set_title('Name Match Rate Distribution', fontsize=13, fontweight='bold')
axes[1, 1].axvline(summary['pct_name_match'].mean(), color='red', linestyle='--',
                    label=f"Mean: {summary['pct_name_match'].mean():.1f}%")
axes[1, 1].legend()

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/distribution_plots.png', dpi=300, bbox_inches='tight')
print(f"✓ Saved plots to {OUTPUT_DIR}/distribution_plots.png")
plt.show()

## 4. Outlier Detection

In [ ]:
# Define outlier thresholds
LOW_CONCORDANCE_THRESHOLD = 95.0  # %

if 'pct_full_length' in summary.columns:
    outliers = summary[summary['pct_full_length'] < LOW_CONCORDANCE_THRESHOLD].copy()
    outliers = outliers.sort_values('pct_full_length')
    
    print(f"Assemblies with < {LOW_CONCORDANCE_THRESHOLD}% full-length concordance:")
    print("=" * 80)
    
    if len(outliers) > 0:
        display(outliers[['n_rbh_pairs', 'pct_full_length', 'pct_name_match', 
                          'frac_ensembl_covered', 'frac_cat_covered']])
        
        # Save outliers
        outliers.to_csv(f'{OUTPUT_DIR}/outlier_assemblies.tsv', sep='\t')
        print(f"\n✓ Saved {len(outliers)} outliers to {OUTPUT_DIR}/outlier_assemblies.tsv")
    else:
        print(f"✓ No outliers found! All assemblies have ≥ {LOW_CONCORDANCE_THRESHOLD}% concordance")

## 5. Biotype Analysis

In [ ]:
# Overall biotype distribution
biotype_counts = master_rbh['ensembl_biotype'].value_counts()

print("Top 10 Biotypes (across all assemblies):")
print("=" * 80)
display(biotype_counts.head(10))

# Plot
plt.figure(figsize=(12, 6))
biotype_counts.head(15).plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Top 15 Gene Biotypes (All Assemblies Combined)', fontsize=14, fontweight='bold')
plt.xlabel('Biotype', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/biotype_distribution.png', dpi=300, bbox_inches='tight')
print(f"✓ Saved plot to {OUTPUT_DIR}/biotype_distribution.png")
plt.show()

In [ ]:
# Concordance by biotype
biotype_concordance = master_rbh.groupby('ensembl_biotype').agg({
    'ensembl_id': 'count',
    'classification': lambda x: (x == 'Full-length concordant').sum() / len(x) * 100,
    'is_name_match': lambda x: x.sum() / len(x) * 100,
    'frac_ensembl_covered': 'mean',
    'frac_cat_covered': 'mean'
}).rename(columns={
    'ensembl_id': 'count',
    'classification': 'pct_full_length',
    'is_name_match': 'pct_name_match'
})

biotype_concordance = biotype_concordance[biotype_concordance['count'] >= 100]  # Filter low-count biotypes
biotype_concordance = biotype_concordance.sort_values('pct_full_length', ascending=False)

print("\nConcordance by Biotype (≥100 instances):")
print("=" * 80)
display(biotype_concordance)

## 6. Classification Breakdown

In [ ]:
# Overall classification counts
classification_totals = master_rbh['classification'].value_counts()

print("Overall Classification Distribution:")
print("=" * 80)
for cls, count in classification_totals.items():
    pct = count / len(master_rbh) * 100
    print(f"  {cls:30s}: {count:,} ({pct:.1f}%)")

# Pie chart
plt.figure(figsize=(10, 8))
colors = ['#2ecc71', '#3498db', '#e74c3c', '#f39c12', '#95a5a6']
classification_totals.plot(kind='pie', autopct='%1.1f%%', startangle=90, 
                            colors=colors[:len(classification_totals)])
plt.title('Classification Distribution (All Assemblies)', fontsize=14, fontweight='bold')
plt.ylabel('')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/classification_pie.png', dpi=300, bbox_inches='tight')
print(f"\n✓ Saved plot to {OUTPUT_DIR}/classification_pie.png")
plt.show()

## 7. Export Master Table

In [ ]:
# Save full master RBH table (can be large!)
master_rbh.to_csv(f'{OUTPUT_DIR}/master_rbh_all_assemblies.tsv.gz', 
                   sep='\t', index=False, compression='gzip')
print(f"✓ Saved master RBH table ({len(master_rbh):,} rows) to {OUTPUT_DIR}/master_rbh_all_assemblies.tsv.gz")

# Save a sample for quick inspection
master_rbh.sample(min(10000, len(master_rbh))).to_csv(
    f'{OUTPUT_DIR}/master_rbh_sample_10k.tsv', sep='\t', index=False
)
print(f"✓ Saved 10k sample to {OUTPUT_DIR}/master_rbh_sample_10k.tsv")

## 8. Final Summary Report

In [ ]:
report = f"""
{'='*80}
HPRC ENSEMBL VS CAT COMPARISON - FINAL SUMMARY
{'='*80}

Assemblies Processed: {len(summary)}
Total RBH Pairs: {len(master_rbh):,}

CONCORDANCE METRICS:
-------------------
Mean Full-length Concordance: {summary['pct_full_length'].mean():.1f}%
Median Full-length Concordance: {summary['pct_full_length'].median():.1f}%
Range: {summary['pct_full_length'].min():.1f}% - {summary['pct_full_length'].max():.1f}%

Mean Name Match Rate: {summary['pct_name_match'].mean():.1f}%

COVERAGE METRICS:
----------------
Mean Ensembl Coverage: {summary['frac_ensembl_covered'].mean():.4f}
Mean CAT Coverage: {summary['frac_cat_covered'].mean():.4f}

OUTLIERS:
---------
Assemblies with <95% concordance: {len(outliers) if 'outliers' in locals() else 0}

TOP BIOTYPES:
------------
{biotype_counts.head(5).to_string()}

OUTPUT FILES:
------------
- {OUTPUT_DIR}/summary_per_assembly.tsv
- {OUTPUT_DIR}/master_rbh_all_assemblies.tsv.gz
- {OUTPUT_DIR}/distribution_plots.png
- {OUTPUT_DIR}/biotype_distribution.png
- {OUTPUT_DIR}/classification_pie.png

{'='*80}
"""

print(report)

# Save report
with open(f'{OUTPUT_DIR}/SUMMARY_REPORT.txt', 'w') as f:
    f.write(report)

print(f"\n✓ Analysis complete! All outputs saved to {OUTPUT_DIR}/")